In [1]:
import pandas as pd
import numpy as np
import optuna
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score

In [2]:
# ---------------------------------------------------------
# 1. Load the Datasets & Filter for Classes 2, 3, 4
# ---------------------------------------------------------
print("Loading datasets...")

DATA_PATH = '../processed_datasets/'
X_TRAIN_PATH = DATA_PATH + 'X_train_catboost.parquet'
X_VAL_PATH = DATA_PATH + 'X_val_catboost.parquet'
X_TEST_PATH = DATA_PATH + 'X_test_catboost.parquet'

Y_TRAIN_PATH = DATA_PATH + 'y_train.csv'
Y_VAL_PATH = DATA_PATH + 'y_val.csv'
Y_TEST_PATH = DATA_PATH + 'y_test.csv'

X_train = pd.read_parquet(X_TRAIN_PATH, engine="fastparquet")
X_val = pd.read_parquet(X_VAL_PATH, engine="fastparquet")
X_test = pd.read_parquet(X_TEST_PATH, engine="fastparquet")

y_train = pd.read_csv(Y_TRAIN_PATH).squeeze()
y_val = pd.read_csv(Y_VAL_PATH).squeeze()
y_test = pd.read_csv(Y_TEST_PATH).squeeze()

# EXCLUSIVE FILTER: Only keep observations for classes 3 and 4
mask_train = y_train >= 3
mask_val = y_val >= 3
mask_test = y_test >= 3

X_train, y_train = X_train[mask_train], y_train[mask_train]
X_val, y_val = X_val[mask_val], y_val[mask_val]
X_test, y_test = X_test[mask_test], y_test[mask_test]

print(f"Filtered for Classes 3, 4.")
print(f"Train shapes: X={X_train.shape}, y={y_train.shape}")
print(f"Validation shapes: X={X_val.shape}, y={y_val.shape}")
print(f"Test shapes: X={X_test.shape}, y={y_test.shape}\n")

categorical_features = X_train.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
categorical_features.extend(['DAY_OF_WEEK'])
print(f"Detected {len(categorical_features)} categorical features.")

Loading datasets...
Filtered for Classes 3, 4.
Train shapes: X=(62556, 32), y=(62556,)
Validation shapes: X=(4337, 32), y=(4337,)
Test shapes: X=(3453, 32), y=(3453,)

Detected 11 categorical features.


In [3]:
# ---------------------------------------------------------
# 2. Define the Optuna Objective Function
# ---------------------------------------------------------
static_params = {
    "random_seed": 42,
    "verbose": False,
    "task_type": "GPU",
    "auto_class_weights": "Balanced",
    "eval_metric": "TotalF1:average=Macro"
}

def objective(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 500, 2000),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.1, log=True),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 10),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 1, 50),
    }
    params.update(static_params)
    
    model = CatBoostClassifier(**params)

    model.fit(
        X_train, 
        y_train,
        cat_features=categorical_features,
        eval_set=[(X_val, y_val)],
        early_stopping_rounds=100, 
        use_best_model=True
    )

    preds_class = np.squeeze(model.predict(X_val))
    macro_f1 = f1_score(y_val, preds_class, average='macro')
    
    return macro_f1

In [4]:
# ---------------------------------------------------------
# 3. Run the Optuna Study
# ---------------------------------------------------------
print("Starting Optuna hyperparameter tuning for Classes 3-4...")

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30) 

print("\n--- Optimization Finished ---")
print(f"Best Macro F1: {study.best_value:.4f}")
print("Best Parameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

[I 2026-06-09 17:35:56,079] A new study created in memory with name: no-name-7ccb9d8a-64fb-4586-81c6-d6e832f1df9a


Starting Optuna hyperparameter tuning for Classes 3-4...


[I 2026-06-09 17:36:37,344] Trial 0 finished with value: 0.6654044894530651 and parameters: {'iterations': 978, 'learning_rate': 0.0018495042514660727, 'depth': 8, 'l2_leaf_reg': 1.8155881907991582, 'min_data_in_leaf': 11}. Best is trial 0 with value: 0.6654044894530651.
[I 2026-06-09 17:36:49,405] Trial 1 finished with value: 0.675657047686614 and parameters: {'iterations': 1661, 'learning_rate': 0.031055502598743005, 'depth': 9, 'l2_leaf_reg': 4.605439464178328, 'min_data_in_leaf': 14}. Best is trial 1 with value: 0.675657047686614.
[I 2026-06-09 17:37:34,769] Trial 2 finished with value: 0.6624825539562232 and parameters: {'iterations': 1687, 'learning_rate': 0.0011853075244800785, 'depth': 8, 'l2_leaf_reg': 5.239459283904061, 'min_data_in_leaf': 47}. Best is trial 1 with value: 0.675657047686614.
[I 2026-06-09 17:37:44,794] Trial 3 finished with value: 0.6718849115129224 and parameters: {'iterations': 1330, 'learning_rate': 0.014259616888222665, 'depth': 4, 'l2_leaf_reg': 4.8725109


--- Optimization Finished ---
Best Macro F1: 0.7007
Best Parameters:
  iterations: 1162
  learning_rate: 0.09745662058770767
  depth: 10
  l2_leaf_reg: 7.65489929550799
  min_data_in_leaf: 1


In [7]:
# ---------------------------------------------------------
# 4. Train Final Model & Evaluate on Test Set
# ---------------------------------------------------------
print("\nTraining final model for Classes 2-4...")

final_params = study.best_params.copy()
final_params.update(static_params)
final_model = CatBoostClassifier(**final_params)

final_model.fit(
    X_train, 
    y_train,
    cat_features=categorical_features,
    eval_set=[(X_val, y_val)],
    early_stopping_rounds=100,
    use_best_model=True
)

y_pred = final_model.predict(X_test)
print(f"Final Test Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred))


Training final model for Classes 2-4...
Final Test Accuracy: 0.7026

Classification Report:
              precision    recall  f1-score   support

           3       0.76      0.79      0.78      2274
           4       0.57      0.53      0.55      1179

    accuracy                           0.70      3453
   macro avg       0.67      0.66      0.66      3453
weighted avg       0.70      0.70      0.70      3453



In [8]:
final_model.save_model("catboost_model_3to4.cbm")